# BitFit 实战

## Step1 导入相关包

In [1]:
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForSeq2Seq,
    TrainingArguments,
    Trainer,
)

## Step2 加载数据集

In [2]:
ds = Dataset.load_from_disk("./data/alpaca_data_zh/")
ds

Dataset({
    features: ['output', 'input', 'instruction'],
    num_rows: 26858
})

In [3]:
ds[:3]

{'output': ['以下是保持健康的三个提示：\n\n1. 保持身体活动。每天做适当的身体运动，如散步、跑步或游泳，能促进心血管健康，增强肌肉力量，并有助于减少体重。\n\n2. 均衡饮食。每天食用新鲜的蔬菜、水果、全谷物和脂肪含量低的蛋白质食物，避免高糖、高脂肪和加工食品，以保持健康的饮食习惯。\n\n3. 睡眠充足。睡眠对人体健康至关重要，成年人每天应保证 7-8 小时的睡眠。良好的睡眠有助于减轻压力，促进身体恢复，并提高注意力和记忆力。',
  '4/16等于1/4是因为我们可以约分分子分母都除以他们的最大公约数4，得到（4÷4）/ (16÷4）=1/4。分数的约分是用分子和分母除以相同的非零整数，来表示分数的一个相同的值，这因为分数实际上表示了分子除以分母，所以即使两个数同时除以同一个非零整数，分数的值也不会改变。所以4/16 和1/4是两种不同的书写形式，但它们的值相等。',
  '朱利叶斯·凯撒，又称尤利乌斯·恺撒（Julius Caesar）是古罗马的政治家、军事家和作家。他于公元前44年3月15日被刺杀。 \n\n根据历史记载，当时罗马元老院里一些参议员联合起来策划了对恺撒的刺杀行动，因为他们担心恺撒的统治将给罗马共和制带来威胁。在公元前44年3月15日（又称“3月的艾达之日”），恺撒去参加元老院会议时，被一群参议员包围并被攻击致死。据记载，他身中23刀，其中一刀最终致命。'],
 'input': ['', '输入：4/16', ''],
 'instruction': ['保持健康的三个提示。', '解释为什么以下分数等同于1/4', '朱利叶斯·凯撒是如何死亡的？']}

## Step3 数据集预处理

In [4]:
tokenizer = AutoTokenizer.from_pretrained("Langboat/bloom-1b4-zh")
tokenizer

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


BloomTokenizerFast(name_or_path='Langboat/bloom-1b4-zh', vocab_size=46145, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [5]:
def process_func(example):
    # print(f"example: {example}")

    MAX_LENGTH = 256
    input_ids, attention_mask, labels = [], [], []

    # print(["Human: " + example["instruction"], example["input"]])
    # print("------------")
    # print("\n".join(["Human: " + example["instruction"], example["input"]]).strip())
    # print("------------")
    # print("\n".join(["Human: " + example["instruction"], example["input"]]).strip() + "\n\nAssistant: ")
    # print("------------")
    # print(example["output"] + tokenizer.eos_token)
    # print("------------" * 10)

    instruction = tokenizer(
        "\n".join(["Human: " + example["instruction"], example["input"]]).strip()
        + "\n\nAssistant: "
    )

    response = tokenizer(example["output"] + tokenizer.eos_token)

    input_ids = instruction["input_ids"] + response["input_ids"]

    attention_mask = instruction["attention_mask"] + response["attention_mask"]

    labels = [-100] * len(instruction["input_ids"]) + response["input_ids"]

    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]

    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

In [6]:
# 调试
# ds = ds.select([0, 3, 1])
# ds

In [7]:
# 调试
# tokenized_ds = ds.map(process_func, remove_columns=ds.column_names)
# tokenized_ds

In [8]:
# 调试
# tokenized_ds[1]

In [9]:
# 调试
# tokenizer.decode(tokenized_ds[1]["input_ids"])

In [10]:
# 调试
# tokenizer.decode(
#     list(
#         filter(lambda x: x != -100, tokenized_ds[1]["labels"])
#     )
# )

In [11]:
tokenized_ds = ds.map(process_func, remove_columns=ds.column_names)
tokenized_ds

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 26858
})

In [12]:
tokenizer.decode(tokenized_ds[1]["input_ids"])

'Human: 解释为什么以下分数等同于1/4\n输入：4/16\n\nAssistant: 4/16等于1/4是因为我们可以约分分子分母都除以他们的最大公约数4，得到（4÷4）/ (16÷4）=1/4。分数的约分是用分子和分母除以相同的非零整数，来表示分数的一个相同的值，这因为分数实际上表示了分子除以分母，所以即使两个数同时除以同一个非零整数，分数的值也不会改变。所以4/16 和1/4是两种不同的书写形式，但它们的值相等。</s>'

In [13]:
tokenizer.decode(list(filter(lambda x: x != -100, tokenized_ds[1]["labels"])))

'4/16等于1/4是因为我们可以约分分子分母都除以他们的最大公约数4，得到（4÷4）/ (16÷4）=1/4。分数的约分是用分子和分母除以相同的非零整数，来表示分数的一个相同的值，这因为分数实际上表示了分子除以分母，所以即使两个数同时除以同一个非零整数，分数的值也不会改变。所以4/16 和1/4是两种不同的书写形式，但它们的值相等。</s>'

## Step4 创建模型

In [14]:
model = AutoModelForCausalLM.from_pretrained(
    "Langboat/bloom-1b4-zh", low_cpu_mem_usage=True
)

In [15]:
# 计算模型的参数
sum(param.numel() for param in model.parameters())

1303111680

model size: 1.3B

model: 1.3G * 4 ~= 5.2G

gradient: 1.3G * 4 ~= 5.2G

optimizer: 1.3G * 4 * 2 ~= 10.4G

sum: 20.8G

## BitFit

In [16]:
# bitfit
# 选择模型参数里面的所有 bias 部分

num_param = 0
for name, param in model.named_parameters():
    print(f"name: {name}")
    if "bias" not in name:
        param.requires_grad = False
    else:
        num_param += param.numel()

num_param

name: transformer.word_embeddings.weight
name: transformer.word_embeddings_layernorm.weight
name: transformer.word_embeddings_layernorm.bias
name: transformer.h.0.input_layernorm.weight
name: transformer.h.0.input_layernorm.bias
name: transformer.h.0.self_attention.query_key_value.weight
name: transformer.h.0.self_attention.query_key_value.bias
name: transformer.h.0.self_attention.dense.weight
name: transformer.h.0.self_attention.dense.bias
name: transformer.h.0.post_attention_layernorm.weight
name: transformer.h.0.post_attention_layernorm.bias
name: transformer.h.0.mlp.dense_h_to_4h.weight
name: transformer.h.0.mlp.dense_h_to_4h.bias
name: transformer.h.0.mlp.dense_4h_to_h.weight
name: transformer.h.0.mlp.dense_4h_to_h.bias
name: transformer.h.1.input_layernorm.weight
name: transformer.h.1.input_layernorm.bias
name: transformer.h.1.self_attention.query_key_value.weight
name: transformer.h.1.self_attention.query_key_value.bias
name: transformer.h.1.self_attention.dense.weight
name: tra

544768

In [17]:
num_param / sum(param.numel() for param in model.parameters())

0.000418051659240749

## Step5 配置训练参数

In [18]:
args = TrainingArguments(
    output_dir="./chatbot",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    logging_steps=10,
    num_train_epochs=1,
)

## Step6 创建训练器

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    tokenizer=tokenizer,
    train_dataset=tokenized_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
)

## Step7 模型训练

In [ ]:
trainer.train()

In [ ]:
model = model.cuda()
ipt = tokenizer(
    "Human: {}\n{}".format("考试有哪些技巧？", "").strip() + "\n\nAssistant: ",
    return_tensors="pt",
).to(model.device)
tokenizer.decode(
    model.generate(**ipt, max_length=128, do_sample=True)[0], skip_special_tokens=True
)

## Step8 模型推理

In [ ]:
from transformers import pipeline

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, device=0)

In [ ]:
ipt = "Human: {}\n{}".format("考试有哪些技巧？", "").strip() + "\n\nAssistant: "
pipe(
    ipt,
    max_length=256,
    do_sample=True,
)